## Gravitational Lens Classification with EfficientNet-B3
#### Task: 3-class image classification on gravitational lensing .npy images.

| Class | Directory | Description |
|---|---|---|
| `no_sub` | `no/` | Smooth lens: no dark matter substructure |
| `subhalo` | `sphere/` | CDM subhalo: localised density perturbations |
| `vortex` | `vort/` | Vortex substructure: coherent angular momentum features |

**Author:** Animesh Parashar  
**Date:** March 2026  
**Trained Model weights:** [Google Drive](https://drive.google.com/file/d/15K96Egu0kP8_AHrwXfA7Oqr191vuIGOU/view?usp=sharing)


### Imports

In [ ]:
import os
import warnings
import random
import time

warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import models
import torchvision.transforms as T
from sklearn.metrics import roc_auc_score, roc_curve, auc as sk_auc
from sklearn.preprocessing import label_binarize
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt

### Reproducibility & Device

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

### Global Configuration

In [ ]:
CFG = dict(
    train_dir    = "/kaggle/input/datasets/stellarquant/deeplensetask1/dataset/train",
    test_dir     = "/kaggle/input/datasets/stellarquant/deeplensetask1/dataset/val",

    ckpt_path    = "/kaggle/working/best_efficientnet_b3.pth",
    history_path = "/kaggle/working/efficientnet_b3_history.npy",
    plot_loss_path    = "/kaggle/working/efficientnet_b3_loss.png",
    plot_auc_path     = "/kaggle/working/efficientnet_b3_auc.png",
    roc_perclass_path = "/kaggle/working/efficientnet_b3_roc_perclass.png",
    roc_macro_path    = "/kaggle/working/efficientnet_b3_roc_macro.png",

    train_frac   = 0.90,
    num_workers  = 4,
    batch_size   = 64,

    epochs       = 80,
    lr           = 3e-4,
    weight_decay = 1e-4,
    warmup_ep    = 3,
    eta_min      = 1e-6,

    tta_n_augs   = 8,

    num_classes  = 3,
    class_names  = ["No Substructure", "Sphere Substructure", "Vortex Substructure"],
    label_smooth = 0.05,

    amp          = True,
    img_size     = 300,
)

IMG_MEAN = [0.0615]
IMG_STD  = [0.1166]

### Dataset

In [ ]:
class LensingDataset(Dataset):
    def __init__(self, root):
        self.files, self.labels = [], []
        self.class_to_idx = {}

        subdirs = sorted([d for d in os.listdir(root)
                          if os.path.isdir(os.path.join(root, d))])
        for idx, cls in enumerate(subdirs):
            self.class_to_idx[cls] = idx
            cls_dir = os.path.join(root, cls)
            for fname in sorted(os.listdir(cls_dir)):
                if fname.endswith(".npy"):
                    self.files.append(os.path.join(cls_dir, fname))
                    self.labels.append(idx)

        print(f"  {len(self.files)} samples  |  root: {root}")
        for k, v in self.class_to_idx.items():
            print(f"    [{v}] {k}: {self.labels.count(v)}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        arr = np.load(self.files[idx]).astype(np.float32)

        if arr.ndim == 2:
            arr = arr[np.newaxis]
        elif arr.ndim == 3 and arr.shape[0] != 1:
            arr = arr[[0]]

        mn, mx = arr.min(), arr.max()
        if mx > mn:
            arr = (arr - mn) / (mx - mn)

        return torch.from_numpy(arr), self.labels[idx]

### Transformed Subset

In [ ]:
class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        x, y = self.subset[i]
        return self.transform(x), y

### Augmentation Pipeline

In [ ]:
def make_train_transform():
    return T.Compose([
        T.Resize((CFG["img_size"], CFG["img_size"]), antialias=True),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(degrees=180),
        T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.90, 1.10)),
        T.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ])


def make_val_transform():
    return T.Compose([
        T.Resize((CFG["img_size"], CFG["img_size"]), antialias=True),
        T.Normalize(mean=IMG_MEAN, std=IMG_STD),
    ])

### Model Definition
#### Model: EfficientNet-B3 (Adapted for Grayscale)
Two modifications are made to the ImageNet baseline:
- **Input conv**: replaced from 3-channel → 1-channel. Pretrained RGB weights are averaged across channels to preserve learned low-level features.
- **Classifier head**: replaced with `Dropout(0.30) → Linear(num_classes)` to reduce overfitting.

In [ ]:
def build_efficientnet_b3(num_classes, pretrained=True):
    weights  = models.EfficientNet_B3_Weights.DEFAULT if pretrained else None
    model    = models.efficientnet_b3(weights=weights)

    old_conv = model.features[0][0]
    new_conv = nn.Conv2d(
        in_channels  = 1,
        out_channels = old_conv.out_channels,
        kernel_size  = old_conv.kernel_size,
        stride       = old_conv.stride,
        padding      = old_conv.padding,
        bias         = False,
    )
    with torch.no_grad():
        if pretrained:
            new_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))
        else:
            nn.init.kaiming_normal_(new_conv.weight, mode="fan_out", nonlinearity="relu")

    model.features[0][0] = new_conv

    in_feat = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.30),
        nn.Linear(in_feat, num_classes),
    )
    return model

### LR Scheduler Warm-up + Cosine Annealing
LR rises linearly from 0 → `base_lr` over the first 3 epochs, then follows a cosine decay down to `eta_min=1e-6`. The warm-up prevents large destabilising updates while the classifier head is still randomly initialised.

In [ ]:
class WarmupCosineScheduler(optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, total_epochs,
                 eta_min=1e-6, last_epoch=-1):
        self.warmup  = warmup_epochs
        self.total   = total_epochs
        self.eta_min = eta_min
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        ep = self.last_epoch
        if ep < self.warmup:
            factor = (ep + 1) / max(1, self.warmup)
        else:
            t = (ep - self.warmup) / max(1, self.total - self.warmup)
            factor = (self.eta_min / self.base_lrs[0] +
                      0.5 * (1.0 - self.eta_min / self.base_lrs[0]) *
                      (1.0 + np.cos(np.pi * t)))
        return [b * factor for b in self.base_lrs]

### Training Helpers

In [ ]:
def safe_probs(logits_tensor):
    lf  = logits_tensor.float().clamp(-88.0, 88.0)
    p   = torch.softmax(lf, dim=1).detach().cpu().numpy()
    bad = ~np.isfinite(p).all(axis=1)
    if bad.any():
        p[bad] = 1.0 / p.shape[1]
    return p


def compute_auc(labels, probs, n_classes):
    y_bin = label_binarize(labels, classes=list(range(n_classes)))
    return roc_auc_score(y_bin, probs, multi_class="ovr", average="macro")


def run_epoch(model, loader, criterion, optimizer, scaler, device, train):
    model.train() if train else model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)

            with autocast(enabled=CFG["amp"]):
                logits = model(imgs)
                loss   = criterion(logits, labels)

            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()

            lv = loss.item()
            if np.isfinite(lv):
                total_loss += lv * imgs.size(0)

            all_probs.append(safe_probs(logits))
            all_labels.append(labels.detach().cpu().numpy())

    all_labels = np.concatenate(all_labels)
    all_probs  = np.concatenate(all_probs)
    avg_loss   = total_loss / len(loader.dataset)
    auc        = compute_auc(all_labels, all_probs, CFG["num_classes"])
    return avg_loss, auc

### Test Time Augmentation (TTA) Utilities
8 deterministic views are constructed from the D4 group (4 rotations × 2 reflections). Softmax probabilities are averaged across all views before computing AUC. This exploits the fact that gravitational lens images carry no orientation information, reliably reducing prediction variance.

In [ ]:
def build_tta_transforms():
    base   = make_val_transform()
    d4_ops = [
        [],
        [T.RandomRotation((90,  90))],
        [T.RandomRotation((180, 180))],
        [T.RandomRotation((270, 270))],
        [T.RandomHorizontalFlip(p=1.0)],
        [T.RandomHorizontalFlip(p=1.0), T.RandomRotation((90, 90))],
        [T.RandomVerticalFlip(p=1.0)],
        [T.RandomVerticalFlip(p=1.0),   T.RandomRotation((90, 90))],
    ]
    return [T.Compose([base] + ops) for ops in d4_ops]


@torch.no_grad()
def plain_inference(model, test_ds_raw, device):
    model.eval()
    ds = TransformedSubset(
        Subset(test_ds_raw, list(range(len(test_ds_raw)))),
        make_val_transform(),
    )
    loader = DataLoader(ds, batch_size=CFG["batch_size"] * 2, shuffle=False,
                        num_workers=CFG["num_workers"], pin_memory=True)
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast(enabled=CFG["amp"]):
            logits = model(imgs)
        all_probs.append(safe_probs(logits))
        all_labels.extend(labels.numpy().tolist())
    return np.concatenate(all_probs, axis=0), np.array(all_labels)


@torch.no_grad()
def tta_inference(model, test_ds_raw, device):
    model.eval()
    tta_transforms = build_tta_transforms()[:CFG["tta_n_augs"]]
    labels         = np.array(test_ds_raw.labels)
    probs_sum      = np.zeros((len(test_ds_raw), CFG["num_classes"]), dtype=np.float32)

    for aug_idx, aug_tf in enumerate(tta_transforms):
        aug_ds = TransformedSubset(
            Subset(test_ds_raw, list(range(len(test_ds_raw)))), aug_tf)
        loader = DataLoader(aug_ds, batch_size=CFG["batch_size"] * 2,
                            shuffle=False, num_workers=CFG["num_workers"],
                            pin_memory=True)
        aug_probs = []
        for imgs, _ in loader:
            imgs = imgs.to(device, non_blocking=True)
            with autocast(enabled=CFG["amp"]):
                logits = model(imgs)
            aug_probs.append(safe_probs(logits))
        probs_sum += np.concatenate(aug_probs, axis=0)
        print(f"    TTA view {aug_idx + 1}/{CFG['tta_n_augs']}  done")

    return probs_sum / CFG["tta_n_augs"], labels

### Plotting Utilities

In [ ]:
def smooth(arr, w=3):
    if w <= 1 or len(arr) < w:
        return arr
    return np.convolve(arr, np.ones(w) / w, mode="same")


def _style_ax_white(ax):
    """Apply white-background styling to a single Axes."""
    ax.set_facecolor("white")
    ax.tick_params(colors="black")
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")
    ax.title.set_color("black")
    for spine in ax.spines.values():
        spine.set_edgecolor("#AAAAAA")
    ax.grid(True, color="#DDDDDD", linewidth=0.8)


def plot_history(history, loss_path, auc_path, smooth_w=3):
    """
    Saves and displays two separate white-background plots:
      1. Training vs Validation Loss
      2. Training vs Validation AUC
    """
    epochs = np.arange(1, len(history["tr_loss"]) + 1)
    C = dict(tr="#E05C2A", va="#1A6BB5")   # warm orange for train, blue for val

    configs = [
        ("tr_loss", "va_loss", "Cross-Entropy Loss",
         "Training and Validation Loss",
         "EfficientNet-B3 — Loss Curve", loss_path),
        ("tr_auc",  "va_auc",  "Macro OvR AUC",
         "Training and Validation AUC (Macro OvR)",
         "EfficientNet-B3 — AUC Curve", auc_path),
    ]

    for key_tr, key_va, ylabel, title, suptitle, save_path in configs:
        fig, ax = plt.subplots(figsize=(8, 5))
        fig.patch.set_facecolor("white")
        _style_ax_white(ax)

        # Faint raw curves
        ax.plot(epochs, history[key_tr], color=C["tr"], alpha=0.25, linewidth=1)
        ax.plot(epochs, history[key_va], color=C["va"], alpha=0.25, linewidth=1)

        # Smoothed curves
        ax.plot(epochs, smooth(np.array(history[key_tr]), smooth_w),
                color=C["tr"], linewidth=2.2, label="Train")
        ax.plot(epochs, smooth(np.array(history[key_va]), smooth_w),
                color=C["va"], linewidth=2.2, linestyle="--", label="Validation")

        ax.set_xlabel("Epoch", fontsize=11, color="black")
        ax.set_ylabel(ylabel, fontsize=11, color="black")
        ax.set_title(title, fontsize=12, fontweight="bold", color="black", pad=10)
        ax.legend(fontsize=10, facecolor="white", edgecolor="#AAAAAA",
                  labelcolor="black", framealpha=1)

        fig.suptitle(suptitle, fontsize=13, fontweight="bold",
                     color="black", y=1.02)
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches="tight",
                    facecolor="white")
        plt.show()
        plt.close(fig)
        print(f"  Saved: {save_path}")


def plot_roc(labels, probs, plain_auc, tta_auc,
             perclass_path, macro_path):
    """
    Saves and displays two separate white-background ROC plots:
      1. Per-class One-vs-Rest ROC curves
      2. Macro-averaged ROC curve with plain vs TTA AUC annotation
    """
    y_bin  = label_binarize(labels, classes=list(range(CFG["num_classes"])))
    COLORS = ["#D62728", "#2CA02C", "#FF7F0E"]   # red, green, orange — visible on white

    # Plot 1: Per-class ROC                                              #
    fig, ax = plt.subplots(figsize=(8, 6))
    fig.patch.set_facecolor("white")
    _style_ax_white(ax)

    per_cls_auc = []
    for c in range(CFG["num_classes"]):
        fpr, tpr, _ = roc_curve(y_bin[:, c], probs[:, c])
        cls_auc     = sk_auc(fpr, tpr)
        per_cls_auc.append(cls_auc)
        ax.plot(fpr, tpr, color=COLORS[c], lw=2,
                label=f"{CFG['class_names'][c]}  (AUC = {cls_auc:.4f})")

    ax.plot([0, 1], [0, 1], color="#999999", lw=1,
            linestyle="--", label="Random classifier")
    ax.set_xlabel("False Positive Rate", fontsize=11, color="black")
    ax.set_ylabel("True Positive Rate",  fontsize=11, color="black")
    ax.set_title("Per-Class ROC Curves (with D4 Test-Time Augmentation)",
                 fontsize=12, fontweight="bold", color="black", pad=10)
    ax.legend(fontsize=9, facecolor="white", edgecolor="#AAAAAA",
              labelcolor="black", framealpha=1)

    fig.suptitle("EfficientNet-B3 — Per-Class ROC",
                 fontsize=13, fontweight="bold", color="black", y=1.02)
    plt.tight_layout()
    plt.savefig(perclass_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    print(f"  Saved: {perclass_path}")

    # Plot 2:Macro-averaged ROC     
    all_fpr = np.unique(np.concatenate([
        roc_curve(y_bin[:, c], probs[:, c])[0]
        for c in range(CFG["num_classes"])
    ]))
    mean_tpr = np.zeros_like(all_fpr)
    for c in range(CFG["num_classes"]):
        fpr, tpr, _ = roc_curve(y_bin[:, c], probs[:, c])
        mean_tpr   += np.interp(all_fpr, fpr, tpr)
    mean_tpr  /= CFG["num_classes"]
    macro_auc  = sk_auc(all_fpr, mean_tpr)

    fig, ax = plt.subplots(figsize=(8, 6))
    fig.patch.set_facecolor("white")
    _style_ax_white(ax)

    ax.plot(all_fpr, mean_tpr, color="#1A6BB5", lw=2.5,
            label=f"Macro-average ROC with TTA  (AUC = {macro_auc:.4f})")
    ax.plot([0, 1], [0, 1], color="#999999", lw=1,
            linestyle="--", label="Random classifier")
    ax.set_xlabel("False Positive Rate", fontsize=11, color="black")
    ax.set_ylabel("True Positive Rate",  fontsize=11, color="black")
    ax.set_title("Macro-Averaged ROC Curve (Equal Weight Across Classes)",
                 fontsize=12, fontweight="bold", color="black", pad=10)
    ax.legend(fontsize=10, facecolor="white", edgecolor="#AAAAAA",
              labelcolor="black", framealpha=1)

    direction = "improvement" if tta_auc >= plain_auc else "drop"
    txt = (f"Plain inference AUC : {plain_auc:.4f}\n"
           f"TTA inference AUC   : {tta_auc:.4f}  "
           f"({direction} of {abs(tta_auc - plain_auc):.4f} vs plain)")
    ax.text(0.50, 0.08, txt, transform=ax.transAxes,
            fontsize=10, color="black",
            bbox=dict(facecolor="white", edgecolor="#AAAAAA",
                      boxstyle="round,pad=0.5"))

    fig.suptitle(
        (f"EfficientNet-B3 — Macro ROC  |  "
         f"D4 TTA ({CFG['tta_n_augs']} views)  |  "
         f"Macro AUC = {macro_auc:.4f}"),
        fontsize=13, fontweight="bold", color="black", y=1.02,
    )
    plt.tight_layout()
    plt.savefig(macro_path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    print(f"  Saved: {macro_path}")

    return per_cls_auc

### Dataset Preparation & DataLoaders

In [ ]:
full_ds = LensingDataset(CFG["train_dir"])

n_total = len(full_ds)
n_train = int(n_total * CFG["train_frac"])
n_val   = n_total - n_train

train_sub, val_sub = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED),
)
print(f"  90/10 split -> train: {n_train}  |  val: {n_val}")

train_ds = TransformedSubset(train_sub, make_train_transform())
val_ds   = TransformedSubset(val_sub,   make_val_transform())

test_ds_raw = LensingDataset(CFG["test_dir"])
test_ds_tf  = TransformedSubset(
    Subset(test_ds_raw, list(range(len(test_ds_raw)))),
    make_val_transform(),
)

train_loader = DataLoader(
    train_ds, batch_size=CFG["batch_size"],
    shuffle=True,  num_workers=CFG["num_workers"],
    pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=CFG["batch_size"] * 2,
    shuffle=False, num_workers=CFG["num_workers"],
    pin_memory=True,
)
test_loader = DataLoader(
    test_ds_tf, batch_size=CFG["batch_size"] * 2,
    shuffle=False, num_workers=CFG["num_workers"],
    pin_memory=True,
)

### Model, Loss, Optimiser, Scheduler, Scaler

In [ ]:
model     = build_efficientnet_b3(CFG["num_classes"], pretrained=True).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smooth"])
optimizer = optim.Adam(model.parameters(), lr=CFG["lr"],
                       weight_decay=CFG["weight_decay"])
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_epochs=CFG["warmup_ep"],
    total_epochs=CFG["epochs"],
    eta_min=CFG["eta_min"],
)
scaler   = GradScaler(enabled=CFG["amp"])
best_auc = 0.0
history  = dict(tr_loss=[], va_loss=[], tr_auc=[], va_auc=[])

### Training Loop

Each epoch: forward pass → label-smoothed cross-entropy loss → AMP-scaled backward → gradient clip (max norm 1.0) → Adam step → cosine LR step. Best checkpoint is saved whenever validation macro OvR AUC improves.

In [ ]:
header = (f"{'Ep':>3} | {'LR':>9} | "
          f"{'Tr Loss':>8} {'Tr AUC':>8} | "
          f"{'Va Loss':>8} {'Va AUC':>8} | "
          f"{'Time(s)':>7} {'Best':>5}")
sep = "-" * len(header)
print(header)
print(sep)

for ep in range(1, CFG["epochs"] + 1):
    t0 = time.time()

    tr_loss, tr_auc = run_epoch(model, train_loader, criterion,
                                optimizer, scaler, DEVICE, train=True)
    va_loss, va_auc = run_epoch(model, val_loader,   criterion,
                                optimizer, scaler, DEVICE, train=False)
    scheduler.step()

    lr      = optimizer.param_groups[0]["lr"]
    elapsed = time.time() - t0
    is_best = va_auc > best_auc
    best_marker = "[best]" if is_best else ""

    history["tr_loss"].append(tr_loss)
    history["va_loss"].append(va_loss)
    history["tr_auc"].append(tr_auc)
    history["va_auc"].append(va_auc)

    print(f"{ep:>3} | {lr:>9.2e} | "
          f"{tr_loss:>8.4f} {tr_auc:>8.4f} | "
          f"{va_loss:>8.4f} {va_auc:>8.4f} | "
          f"{elapsed:>7.0f} {best_marker}")

    if is_best:
        best_auc = va_auc
        torch.save({
            "epoch":                ep,
            "val_auc":              float(va_auc),
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        }, CFG["ckpt_path"])

    torch.cuda.empty_cache()

print(sep)
print(f"Best validation AUC = {best_auc:.4f}  (checkpoint saved to {CFG['ckpt_path']})")

### Test-Set Evaluation (Plain + TTA)

The best checkpoint is reloaded and evaluated twice, once with standard inference and once with D4 TTA; to quantify the AUC gain from augmentation averaging.

In [ ]:
np.save(CFG["history_path"], history)
plot_history(history, CFG["plot_loss_path"], CFG["plot_auc_path"], smooth_w=3)

ckpt = torch.load(CFG["ckpt_path"], map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
print(f"  Loaded checkpoint from epoch {ckpt['epoch']}  |  val AUC = {ckpt['val_auc']:.4f}")

plain_probs, plain_labels = plain_inference(model, test_ds_raw, DEVICE)
y_bin_plain  = label_binarize(plain_labels, classes=list(range(CFG["num_classes"])))
te_auc_plain = roc_auc_score(y_bin_plain, plain_probs,
                              multi_class="ovr", average="macro")
print(f"  Plain inference AUC (no TTA) : {te_auc_plain:.4f}")

print(f"\nRunning D4 test-time augmentation ({CFG['tta_n_augs']} views) ...")
tta_probs_arr, tta_labels = tta_inference(model, test_ds_raw, DEVICE)
y_bin_tta  = label_binarize(tta_labels, classes=list(range(CFG["num_classes"])))
te_auc_tta = roc_auc_score(y_bin_tta, tta_probs_arr,
                            multi_class="ovr", average="macro")

direction = "improvement" if te_auc_tta >= te_auc_plain else "drop"
print(f"\n  Plain inference AUC (no TTA)                    : {te_auc_plain:.4f}")
print(f"  TTA inference AUC  ({CFG['tta_n_augs']} D4 views)             : {te_auc_tta:.4f}  "
      f"({direction} of {abs(te_auc_tta - te_auc_plain):.4f} vs plain)")
print("\n  Per-class AUC breakdown (TTA):")
for c, name in enumerate(CFG["class_names"]):
    cls_auc = roc_auc_score(y_bin_tta[:, c], tta_probs_arr[:, c])
    print(f"    {name:<28}: {cls_auc:.4f}")

plot_roc(tta_labels, tta_probs_arr, te_auc_plain, te_auc_tta,
         CFG["roc_perclass_path"], CFG["roc_macro_path"])